# Machine Learning Assignment 2 - Colab Solution

**Course:** Machine Learning - M.Tech (AIML/DSE)  
**Assignment:** Classification models and Streamlit web application  
**Dataset:** Breast Cancer Wisconsin Diagnostic dataset

This notebook is self-contained for Google Colab. Run the cells from top to bottom. It trains all models, computes the required metrics, saves `test_data.csv`, writes the Streamlit `app.py`, creates saved model artifacts, and provides a cell to launch the Streamlit app from Colab.

The assignment instruction is treated as updated: **Using AI tools is completely allowed.**

Before final submission, replace the placeholder GitHub and Streamlit links in the generated README/report text with your actual links after deployment.


## What this notebook produces

After running all cells, Colab will contain a folder named `ml_assignment_2_breast_cancer_classifier` with:

- `app.py` - Streamlit app
- `requirements.txt` - dependencies for Streamlit Community Cloud
- `README.md` - assignment report content
- `test_data.csv` - held-out test split
- `model/*.joblib` - saved trained models
- `model/model_metadata.json` - model paths, feature names, metrics, and metadata
- `model_metrics.csv` - comparison metrics table
- `ml_assignment_2_breast_cancer_classifier.zip` - downloadable project zip

The notebook itself also displays the model comparison table, observations, and confusion matrix for the winning model.


In [ ]:
# Colab dependency setup
# Installs packages into the exact Python interpreter running this notebook.
# This is more reliable than shell/pip magics in managed notebook runtimes.

import subprocess
import sys

packages = [
    "streamlit",
    "scikit-learn",
    "pandas",
    "numpy",
    "joblib",
]

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])
print("Dependencies installed for:", sys.executable)


In [ ]:
from __future__ import annotations

import json
import shutil
import subprocess
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42
PROJECT_DIR = Path("ml_assignment_2_breast_cancer_classifier")
MODEL_DIR = PROJECT_DIR / "model"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project directory: {PROJECT_DIR.resolve()}")


## Step 1 - Dataset choice

The selected dataset is the **Breast Cancer Wisconsin Diagnostic** dataset, originally from the UCI Machine Learning Repository and available directly through scikit-learn.

It satisfies the assignment constraints:

- Minimum feature size required: 12; selected dataset has 30 features.
- Minimum instance size required: 500; selected dataset has 569 instances.
- Problem type: binary classification.
- Target classes: benign and malignant.

For evaluation metrics, malignant is treated as the positive class (`1`) and benign as class `0`.


In [ ]:
def load_assignment_dataset() -> tuple[pd.DataFrame, pd.Series, dict]:
    dataset = load_breast_cancer()
    features = pd.DataFrame(dataset.data, columns=dataset.feature_names)

    # scikit-learn stores malignant as 0 and benign as 1.
    # For this assignment, 1 is used as the positive class: malignant.
    target = pd.Series(np.where(dataset.target == 0, 1, 0), name="diagnosis")

    metadata = {
        "name": "Breast Cancer Wisconsin Diagnostic",
        "source": "UCI Machine Learning Repository, available through scikit-learn",
        "samples": int(features.shape[0]),
        "features": int(features.shape[1]),
        "problem_type": "Binary classification",
        "target_mapping": {"0": "Benign", "1": "Malignant"},
        "positive_class": "Malignant",
        "feature_names": list(features.columns),
    }
    return features, target, metadata


X, y, metadata = load_assignment_dataset()
full_dataset_preview = X.copy()
full_dataset_preview["diagnosis"] = y.map({0: "Benign", 1: "Malignant"})

print(f"Rows: {X.shape[0]}")
print(f"Features: {X.shape[1]}")
print("Class distribution:")
print(full_dataset_preview["diagnosis"].value_counts())
display(full_dataset_preview.head())


## Step 2 - Model implementation and evaluation metrics

The assignment text says six models are required, while the visible list/table names five models. This notebook implements the five listed models and adds **Support Vector Machine** as the sixth standard classifier.

Models implemented:

1. Logistic Regression
2. Decision Tree Classifier
3. k-Nearest Neighbor Classifier
4. Naive Bayes Classifier
5. Random Forest Classifier
6. Support Vector Machine

Metrics calculated for each model:

- Accuracy
- AUC score
- Precision
- Recall
- F1 score
- Matthews Correlation Coefficient (MCC)


In [ ]:
def build_models() -> dict[str, object]:
    return {
        "Logistic Regression": Pipeline(
            steps=[
                ("scaler", StandardScaler()),
                (
                    "model",
                    LogisticRegression(
                        max_iter=5000,
                        solver="liblinear",
                        random_state=RANDOM_STATE,
                    ),
                ),
            ]
        ),
        "Decision Tree": DecisionTreeClassifier(
            max_depth=5,
            min_samples_leaf=4,
            random_state=RANDOM_STATE,
        ),
        "kNN": Pipeline(
            steps=[
                ("scaler", StandardScaler()),
                ("model", KNeighborsClassifier(n_neighbors=7)),
            ]
        ),
        "Naive Bayes": GaussianNB(),
        "Random Forest (Ensemble)": RandomForestClassifier(
            n_estimators=300,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=1,
        ),
        "Support Vector Machine": Pipeline(
            steps=[
                ("scaler", StandardScaler()),
                (
                    "model",
                    SVC(
                        kernel="rbf",
                        C=2.0,
                        gamma="scale",
                        probability=True,
                        random_state=RANDOM_STATE,
                    ),
                ),
            ]
        ),
    }


def slugify_model_name(name: str) -> str:
    return (
        name.lower()
        .replace("(", "")
        .replace(")", "")
        .replace(" ", "_")
        .replace("-", "_")
    )


def positive_class_scores(model: object, features: pd.DataFrame) -> np.ndarray:
    if hasattr(model, "predict_proba"):
        return model.predict_proba(features)[:, 1]
    if hasattr(model, "decision_function"):
        raw_scores = model.decision_function(features)
        return (raw_scores - raw_scores.min()) / (raw_scores.max() - raw_scores.min())
    return model.predict(features)


def evaluate_model(model: object, features: pd.DataFrame, target: pd.Series) -> dict:
    predictions = model.predict(features)
    scores = positive_class_scores(model, features)
    matrix = confusion_matrix(target, predictions, labels=[0, 1])

    return {
        "Accuracy": round(float(accuracy_score(target, predictions)), 4),
        "AUC": round(float(roc_auc_score(target, scores)), 4),
        "Precision": round(float(precision_score(target, predictions, zero_division=0)), 4),
        "Recall": round(float(recall_score(target, predictions, zero_division=0)), 4),
        "F1": round(float(f1_score(target, predictions, zero_division=0)), 4),
        "MCC": round(float(matthews_corrcoef(target, predictions)), 4),
        "confusion_matrix": matrix.tolist(),
        "classification_report": classification_report(
            target,
            predictions,
            target_names=["Benign", "Malignant"],
            digits=4,
            zero_division=0,
            output_dict=True,
        ),
    }


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

# Save the held-out test data used in the Streamlit app.
test_data = X_test.copy()
test_data["diagnosis"] = y_test.map({0: "Benign", 1: "Malignant"}).values
test_data.to_csv(PROJECT_DIR / "test_data.csv", index=False)

models = build_models()
metrics: dict[str, dict] = {}
artifact_map: dict[str, str] = {}

for model_name, model in models.items():
    model.fit(X_train, y_train)
    metrics[model_name] = evaluate_model(model, X_test, y_test)
    artifact_name = f"{slugify_model_name(model_name)}.joblib"
    artifact_path = MODEL_DIR / artifact_name
    joblib.dump(model, artifact_path)
    artifact_map[model_name] = f"model/{artifact_name}"

metrics_df = pd.DataFrame(
    [
        {
            "ML Model Name": model_name,
            "Accuracy": values["Accuracy"],
            "AUC": values["AUC"],
            "Precision": values["Precision"],
            "Recall": values["Recall"],
            "F1": values["F1"],
            "MCC": values["MCC"],
        }
        for model_name, values in metrics.items()
    ]
)
metrics_df.to_csv(PROJECT_DIR / "model_metrics.csv", index=False)

def aggregate_score(row: pd.Series) -> float:
    return float(row[["Accuracy", "AUC", "Precision", "Recall", "F1", "MCC"]].mean())

metrics_df["Aggregate Score"] = metrics_df.apply(aggregate_score, axis=1)
winner = metrics_df.sort_values("Aggregate Score", ascending=False).iloc[0]["ML Model Name"]
metrics_df = metrics_df.drop(columns=["Aggregate Score"])

metadata.update(
    {
        "random_state": RANDOM_STATE,
        "test_size": 0.2,
        "models": artifact_map,
        "metrics": metrics,
        "winner": winner,
    }
)
(MODEL_DIR / "model_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")

print(f"Saved test data: {PROJECT_DIR / 'test_data.csv'}")
print(f"Saved model artifacts under: {MODEL_DIR}")
print(f"Overall winner: {winner}")
display(metrics_df)


## Step 3 - Observations

The table below gives model-specific observations and the overall winner for this dataset.


In [ ]:
observations = {
    "Logistic Regression": "Strong baseline with high AUC and balanced precision/recall, showing that the standardized feature space is close to linearly separable.",
    "Decision Tree": "Useful and interpretable, but weaker than the best models because a single tree is sensitive to split choices and can overfit local patterns.",
    "kNN": "Performed well after scaling because similar biopsy profiles tend to share the same diagnosis, though it is less transparent than tree or logistic models.",
    "Naive Bayes": "Competitive despite its strong feature-independence assumption, indicating that several individual measurements are highly informative.",
    "Random Forest (Ensemble)": "Best tree-based model because averaging many trees reduced overfitting and improved generalization compared with a single decision tree.",
    "Support Vector Machine": "Best overall model on this split, with the highest aggregate metric score and especially strong accuracy, F1, and MCC after feature scaling.",
}

observation_df = pd.DataFrame(
    [{"ML Model Name": model_name, "Observation about model performance": observations[model_name]} for model_name in metrics]
)
observation_df.loc[len(observation_df)] = {
    "ML Model Name": "Overall Winner for this dataset",
    "Observation about model performance": f"{winner} is the overall winner because it has the strongest aggregate score across accuracy, AUC, precision, recall, F1, and MCC on the held-out test split.",
}

display(observation_df)


In [ ]:
winner_matrix = np.array(metrics[winner]["confusion_matrix"])
confusion_df = pd.DataFrame(
    winner_matrix,
    index=["Actual Benign", "Actual Malignant"],
    columns=["Predicted Benign", "Predicted Malignant"],
)
print(f"Confusion Matrix - {winner}")
display(confusion_df)

winner_report = pd.DataFrame(metrics[winner]["classification_report"]).transpose()
display(winner_report)


## Step 4 - Create `requirements.txt`, `README.md`, and the Streamlit app

The next cell writes all repository files needed for Streamlit Community Cloud deployment. These files are generated inside the Colab runtime from this single notebook.


In [ ]:
requirements_text = """streamlit
scikit-learn
numpy
pandas
joblib
"""
(PROJECT_DIR / "requirements.txt").write_text(requirements_text, encoding="utf-8")

metric_rows = "\n".join(
    "| {model} | {accuracy:.4f} | {auc:.4f} | {precision:.4f} | {recall:.4f} | {f1:.4f} | {mcc:.4f} |".format(
        model=row["ML Model Name"],
        accuracy=row["Accuracy"],
        auc=row["AUC"],
        precision=row["Precision"],
        recall=row["Recall"],
        f1=row["F1"],
        mcc=row["MCC"],
    )
    for _, row in metrics_df.iterrows()
)

observation_rows = "\n".join(
    f"| {row['ML Model Name']} | {row['Observation about model performance']} |"
    for _, row in observation_df.iterrows()
)

readme_text = f"""# Breast Cancer Classification Streamlit App

## Problem statement

The goal of this project is to build an end-to-end machine learning classification application that predicts whether a breast tumor sample is benign or malignant from numeric diagnostic measurements. The project trains multiple classifiers on the same dataset, compares them with standard evaluation metrics, and exposes the results through an interactive Streamlit application.

## Dataset description

- Dataset: {metadata['name']}
- Source: {metadata['source']}
- Problem type: {metadata['problem_type']}
- Number of instances: {metadata['samples']}
- Number of input features: {metadata['features']}
- Target classes: 0 = Benign, 1 = Malignant
- Positive class used for precision, recall, F1, AUC, and MCC: Malignant

The dataset satisfies the assignment constraints because it has more than 500 records and more than 12 features. The bundled `test_data.csv` file contains the held-out test split used for app demonstration and evaluation.

## GitHub repository link

Add your GitHub repository link here after uploading this folder:

`https://github.com/<your-user>/<your-repo>`

## Live Streamlit app link

Add your deployed Streamlit Community Cloud link here after deployment:

`https://<your-app-name>.streamlit.app`

## Models used

The assignment PDF text says six ML models are required, while the visible list/table names five. This solution implements the five listed models and adds Support Vector Machine as the sixth standard classifier.

| ML Model Name | Accuracy | AUC | Precision | Recall | F1 | MCC |
|---|---:|---:|---:|---:|---:|---:|
{metric_rows}

## Observations about model performance

| ML Model Name | Observation about model performance |
|---|---|
{observation_rows}

## How to run locally

```bash
python -m venv .venv
source .venv/bin/activate
pip install -r requirements.txt
streamlit run app.py
```

## Streamlit app features

- CSV upload option for test data
- Model selection dropdown for individual model evaluation
- Comparison table for all models
- Accuracy, AUC, precision, recall, F1 score, and MCC
- Confusion matrix and classification report
- Single-record prediction form

## Repository structure

```text
project-folder/
|-- app.py
|-- requirements.txt
|-- README.md
|-- test_data.csv
|-- model/
|   |-- model_metadata.json
|   |-- *.joblib
|-- model_metrics.csv
```
"""

(PROJECT_DIR / "README.md").write_text(readme_text, encoding="utf-8")

print("Wrote requirements.txt and README.md")


In [ ]:
app_code = '\nfrom __future__ import annotations\n\nimport json\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nimport streamlit as st\nfrom sklearn.metrics import (\n    accuracy_score,\n    classification_report,\n    confusion_matrix,\n    f1_score,\n    matthews_corrcoef,\n    precision_score,\n    recall_score,\n    roc_auc_score,\n)\n\nROOT = Path(__file__).resolve().parent\nMETADATA_PATH = ROOT / "model" / "model_metadata.json"\nDEFAULT_TEST_DATA = ROOT / "test_data.csv"\n\nst.set_page_config(page_title="Breast Cancer Classifier", layout="wide")\n\n\n@st.cache_resource\ndef load_metadata() -> dict:\n    with METADATA_PATH.open("r", encoding="utf-8") as handle:\n        return json.load(handle)\n\n\n@st.cache_resource\ndef load_models(model_paths: dict[str, str]) -> dict[str, object]:\n    return {model_name: joblib.load(ROOT / relative_path) for model_name, relative_path in model_paths.items()}\n\n\ndef normalize_target(series: pd.Series) -> pd.Series:\n    if pd.api.types.is_numeric_dtype(series):\n        return series.astype(int)\n    cleaned = series.astype(str).str.strip().str.lower()\n    mapping = {\n        "benign": 0,\n        "b": 0,\n        "0": 0,\n        "malignant": 1,\n        "m": 1,\n        "1": 1,\n    }\n    return cleaned.map(mapping)\n\n\ndef find_target_column(dataframe: pd.DataFrame) -> str | None:\n    for candidate in ["diagnosis", "target", "label", "class"]:\n        if candidate in dataframe.columns:\n            return candidate\n    return None\n\n\ndef score_positive_class(model: object, features: pd.DataFrame) -> np.ndarray:\n    if hasattr(model, "predict_proba"):\n        return model.predict_proba(features)[:, 1]\n    if hasattr(model, "decision_function"):\n        scores = model.decision_function(features)\n        return (scores - scores.min()) / (scores.max() - scores.min())\n    return model.predict(features)\n\n\ndef prepare_uploaded_data(dataframe: pd.DataFrame, feature_names: list[str]) -> tuple[pd.DataFrame, pd.Series | None]:\n    missing_features = [name for name in feature_names if name not in dataframe.columns]\n    if missing_features:\n        st.error("The uploaded CSV is missing required feature columns.")\n        st.dataframe(pd.DataFrame({"Missing feature": missing_features}), use_container_width=True)\n        st.stop()\n\n    features = dataframe[feature_names].copy()\n    target_column = find_target_column(dataframe)\n    target = None\n    if target_column:\n        target = normalize_target(dataframe[target_column])\n        if target.isna().any():\n            st.error(f"The target column `{target_column}` contains values that could not be mapped to Benign/Malignant.")\n            st.stop()\n    return features, target\n\n\ndef show_metric_cards(metrics: dict) -> None:\n    columns = st.columns(6)\n    for column, metric_name in zip(columns, ["Accuracy", "AUC", "Precision", "Recall", "F1", "MCC"]):\n        column.metric(metric_name, f"{metrics[metric_name]:.4f}")\n\n\ndef show_confusion_matrix(matrix: np.ndarray) -> None:\n    confusion = pd.DataFrame(\n        matrix,\n        index=["Actual Benign", "Actual Malignant"],\n        columns=["Predicted Benign", "Predicted Malignant"],\n    )\n    st.dataframe(confusion, use_container_width=True)\n\n\ndef main() -> None:\n    metadata = load_metadata()\n    models = load_models(metadata["models"])\n    feature_names = metadata["feature_names"]\n\n    st.title("Breast Cancer Classification")\n    st.caption("Interactive evaluation app for Logistic Regression, Decision Tree, kNN, Naive Bayes, Random Forest, and SVM.")\n\n    with st.sidebar:\n        st.header("Dataset")\n        uploaded_file = st.file_uploader("Upload test CSV", type=["csv"])\n        selected_model = st.selectbox("Select model", list(models.keys()))\n        threshold = st.slider("Malignant probability threshold", min_value=0.10, max_value=0.90, value=0.50, step=0.05)\n\n    raw_data = pd.read_csv(uploaded_file) if uploaded_file is not None else pd.read_csv(DEFAULT_TEST_DATA)\n    features, target = prepare_uploaded_data(raw_data, feature_names)\n\n    overview_tab, evaluate_tab, predict_tab = st.tabs(["Model Comparison", "Uploaded Test Data", "Single Prediction"])\n\n    with overview_tab:\n        st.subheader("Dataset summary")\n        col_a, col_b, col_c = st.columns(3)\n        col_a.metric("Rows in current test data", f"{len(features):,}")\n        col_b.metric("Input features", f"{len(feature_names):,}")\n        col_c.metric("Positive class", "Malignant")\n\n        st.subheader("Training-time model comparison")\n        metric_rows = []\n        for model_name, values in metadata["metrics"].items():\n            metric_rows.append(\n                {\n                    "ML Model Name": model_name,\n                    "Accuracy": values["Accuracy"],\n                    "AUC": values["AUC"],\n                    "Precision": values["Precision"],\n                    "Recall": values["Recall"],\n                    "F1": values["F1"],\n                    "MCC": values["MCC"],\n                }\n            )\n        st.dataframe(pd.DataFrame(metric_rows), use_container_width=True, hide_index=True)\n        st.info(f"Overall winner on the held-out split: {metadata[\'winner\']}")\n\n    with evaluate_tab:\n        st.subheader(selected_model)\n        model = models[selected_model]\n        scores = score_positive_class(model, features)\n        predictions = (scores >= threshold).astype(int)\n        result_frame = raw_data.copy()\n        result_frame["predicted_diagnosis"] = np.where(predictions == 1, "Malignant", "Benign")\n        result_frame["malignant_probability"] = scores\n\n        if target is not None:\n            metrics = {\n                "Accuracy": accuracy_score(target, predictions),\n                "AUC": roc_auc_score(target, scores),\n                "Precision": precision_score(target, predictions, zero_division=0),\n                "Recall": recall_score(target, predictions, zero_division=0),\n                "F1": f1_score(target, predictions, zero_division=0),\n                "MCC": matthews_corrcoef(target, predictions),\n                "Confusion Matrix": confusion_matrix(target, predictions, labels=[0, 1]),\n                "Classification Report": classification_report(target, predictions, target_names=["Benign", "Malignant"], digits=4, zero_division=0, output_dict=True),\n            }\n            show_metric_cards(metrics)\n            left, right = st.columns([1, 2])\n            with left:\n                st.subheader("Confusion matrix")\n                show_confusion_matrix(metrics["Confusion Matrix"])\n            with right:\n                st.subheader("Classification report")\n                st.dataframe(pd.DataFrame(metrics["Classification Report"]).transpose(), use_container_width=True)\n        else:\n            st.warning("No target column was found, so the app is showing predictions only.")\n\n        st.subheader("Prediction preview")\n        preview_columns = ["predicted_diagnosis", "malignant_probability"] + feature_names[:6]\n        st.dataframe(result_frame[preview_columns].head(25), use_container_width=True)\n\n    with predict_tab:\n        st.subheader("Predict one record")\n        selected_single_model = st.selectbox("Prediction model", list(models.keys()), key="single_model")\n        inputs = {}\n        columns = st.columns(3)\n        for index, feature_name in enumerate(feature_names):\n            values = features[feature_name]\n            inputs[feature_name] = columns[index % 3].number_input(\n                feature_name,\n                min_value=float(values.min()),\n                max_value=float(values.max()),\n                value=float(values.median()),\n                step=max(float(values.std() / 10), 0.001),\n            )\n\n        single_features = pd.DataFrame([inputs], columns=feature_names)\n        single_model = models[selected_single_model]\n        probability = float(score_positive_class(single_model, single_features)[0])\n        diagnosis = "Malignant" if probability >= threshold else "Benign"\n        st.metric("Predicted diagnosis", diagnosis)\n        st.metric("Malignant probability", f"{probability:.4f}")\n\n\nif __name__ == "__main__":\n    main()\n'
(PROJECT_DIR / "app.py").write_text(app_code, encoding="utf-8")
print(f"Wrote Streamlit app: {PROJECT_DIR / 'app.py'}")


## Step 5 - Sanity checks

This cell confirms that the notebook generated the expected files and that each saved model can make predictions on the held-out test data.


In [ ]:
expected_files = [
    PROJECT_DIR / "app.py",
    PROJECT_DIR / "requirements.txt",
    PROJECT_DIR / "README.md",
    PROJECT_DIR / "test_data.csv",
    PROJECT_DIR / "model_metrics.csv",
    MODEL_DIR / "model_metadata.json",
]
expected_files += [PROJECT_DIR / path for path in artifact_map.values()]

missing = [str(path) for path in expected_files if not path.exists()]
assert not missing, f"Missing generated files: {missing}"
assert len(metrics_df) == 6, "Expected exactly six model rows."
assert X.shape[0] >= 500, "Dataset must have at least 500 rows."
assert X.shape[1] >= 12, "Dataset must have at least 12 features."

loaded_metadata = json.loads((MODEL_DIR / "model_metadata.json").read_text(encoding="utf-8"))
loaded_test_data = pd.read_csv(PROJECT_DIR / "test_data.csv")
for model_name, relative_path in loaded_metadata["models"].items():
    model = joblib.load(PROJECT_DIR / relative_path)
    predictions = model.predict(loaded_test_data[loaded_metadata["feature_names"]])
    assert len(predictions) == len(loaded_test_data)

print("All sanity checks passed.")
print(f"Generated files are under: {PROJECT_DIR.resolve()}")


## Step 6 - Run the Streamlit app on localhost

Run the next cell after all previous cells are complete. It starts Streamlit on your local runtime at:

`http://localhost:8501`

This is the mode to use in BITS Virtual Lab or any local notebook environment. No Cloudflare, LocalTunnel, or external tunnel is used.


In [ ]:
# Start Streamlit locally. No Cloudflare, LocalTunnel, or external tunnel is used.
# Open http://localhost:8501 after this cell reports that Streamlit is running.

import importlib.util
import subprocess
import sys
import time

if importlib.util.find_spec("streamlit") is None:
    print("Streamlit is not installed in this kernel. Installing it now...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "streamlit"])

streamlit_log_path = PROJECT_DIR / "streamlit_local.log"
streamlit_log = open(streamlit_log_path, "w", encoding="utf-8")

streamlit_process = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "streamlit",
        "run",
        str(PROJECT_DIR / "app.py"),
        "--server.address=127.0.0.1",
        "--server.port=8501",
        "--server.headless=true",
    ],
    stdout=streamlit_log,
    stderr=subprocess.STDOUT,
)
time.sleep(6)

if streamlit_process.poll() is not None:
    streamlit_log.close()
    print("Streamlit failed to start. Log output:")
    print(streamlit_log_path.read_text(encoding="utf-8"))
else:
    print("Streamlit is running locally.")
    print("Open this URL in your browser:")
    print("http://localhost:8501")


## Step 7 - Create and download the project zip

Run this after the files are generated. The zip contains all files needed for GitHub and Streamlit Community Cloud deployment.


In [ ]:
zip_path = shutil.make_archive(PROJECT_DIR.name, "zip", PROJECT_DIR)
print(f"Created zip: {zip_path}")

try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    print("Download helper is available only in Google Colab. Download the zip from the file browser if running elsewhere.")
